In [ ]:
import os
import pandas as pd
import numpy as np
from analytics_core.analytics import analytics
from graphdb_connector import connector

In [ ]:
def run_enrichment(go_annotation_df, foreground_list, background_list, foreground_pop, background_pop):
    '''
    param go_annotation_df: pandas dataframe with annotations for features (columns: 'annotation', 'identifier' (feature identifiers), and 'source').
    param foreground_list: list of proteins of interest (e.g. regulated);
    param background_list: list of proteins to calculate the enrichment against;
    param foreground_pop: integer size(len) of the foreground list;
    param background_pop: integer size(len) of the background list;
    '''
    annotation_df = go_annotation_df.copy()
    grouping = []
    for i in annotation_df['identifier']:
        if i in foreground_list:
            grouping.append('foreground')
        elif i in background_list:
            grouping.append('background')
        else:
            grouping.append(np.nan)
    annotation_df['group'] = grouping
    annotation_df = annotation_df.dropna(subset=['group'])
    result = analytics.run_enrichment(annotation_df, foreground_id='foreground', background_id='background', foreground_pop=foreground_pop, background_pop=background_pop, annotation_col='annotation', group_col='group', identifier_col='identifier', method='fisher', correction='fdr_bh')
    return result

### Import proteomics data

### Load significantly regulated proteomics data given a mutated drug state in the VAE

In [ ]:
DATAFOLDER = 'projects/xxx/drug_assosiations_proteomics'
files = [file for file in os.listdir(DATAFOLDER) if '.txt' in file]

In [ ]:
file_names = [i.split('.')[0].split('_')[1] for i in files]
file_paths = [os.path.join(DATAFOLDER, DATA) for DATA in files]
files_data = [pd.read_csv(f_data, sep='\t', header=None) for f_data in file_paths]
for df in files_data:
    df.columns=['Gene name']
files_dict = dict(zip(file_names, files_data))
file_names

In [ ]:
for i in files_data:
    print (i.shape)

### Load background proteomics data 
(all proteins selectively measured by a combination of various immunoassay-based platforms, [reference here](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7304567/))

In [ ]:
background_proteomics = pd.read_csv('projects/xxx/proteomics_incl.txt', sep = '\t', header=None)
background_proteomics.columns=['Gene name']
BACKGROUND_PROTEINS = background_proteomics['Gene name'].unique().tolist()

### Query CKG database for protein annotations

### Before start:
- have CKG running in Neo4j. 
- make sure that the password you defined for it in Neo4j is the same you have defined in CKG/src/graphdb_connector/connector_config.yml Password in Neo4j, go to CKG, under Administration--> set password

In [ ]:
from graphdb_connector import query_utils, connector

##### We connect to CKG database using the default configuration

##### We load the existing database queries that we can use to extract knowledge from CKG

In [ ]:
driver = connector.getGraphDatabaseConnectionConfiguration()
queries = query_utils.read_knowledge_queries()
selected_queries = query_utils.find_queries_involving_nodes(queries=queries, nodes=["Protein", "Drug"], print_pretty=True)
selected_queries.head()

### If you only have gene names, instead of querying for (p.name+'~'+p.id), you would just do p.name

In [ ]:
annotation_query_bp = 'MATCH (p:Protein)-[r:ASSOCIATED_WITH]-(bp:Biological_process) WHERE (p.name) IN LIST RETURN DISTINCT (p.name) AS identifier, bp.name AS annotation'
annotation_query_mf = 'MATCH (p:Protein)-[r:ASSOCIATED_WITH]-(bp:Molecular_function) WHERE (p.name) IN LIST RETURN DISTINCT (p.name) AS identifier, bp.name AS annotation'
annotation_query_drug = 'MATCH (protein:Protein)-[r:ACTS_ON]-(drug:Drug) WHERE (protein.name) IN LIST AND toFloat(r.score)>=0.2 RETURN (protein.name) AS identifier, drug.name AS annotation, drug.id AS drug_id, drug.description AS Drug_desc, r.score AS weight, type(r) AS type, r.action AS action, r.source AS source\n'
annotation_query_pathway = 'MATCH (protein:Protein)-[rp:ANNOTATED_IN_PATHWAY]->(pathway:Pathway) WHERE (protein.name) IN LIST AND toLower(rp.source) IN ["reactome"] RETURN DISTINCT protein.name AS identifier,pathway.name AS annotation,rp.source AS source'
annotation_query_disease = 'MATCH (p:Protein)-[r:ASSOCIATED_WITH]->(d:Disease)-[:HAS_PARENT*2..4]->(pd:Disease) WHERE (p.name) IN LIST AND pd.name="disease of metabolism" RETURN (p.name) AS identifier, d.name AS annotation, d.id AS disease_id, r.score AS score'
annotation_query_complexes= 'MATCH (p:Protein)-[r:IS_SUBUNIT_OF]->(c:Complex) WHERE (p.name) IN LIST RETURN (p.name) AS identifier, c.name AS annotation, r.score AS score'

### Replace "LIST" in the query with all the proteins (background + foreground):

In [ ]:
annotation_query_bp = annotation_query_bp.replace('LIST', str(BACKGROUND_PROTEINS))
annotation_query_mf = annotation_query_mf.replace('LIST', str(BACKGROUND_PROTEINS))
annotation_query_pathway = annotation_query_pathway.replace('LIST', str(BACKGROUND_PROTEINS))
annotation_query_drug = annotation_query_drug.replace('LIST', str(BACKGROUND_PROTEINS))
annotation_query_disease = annotation_query_disease.replace('LIST', str(BACKGROUND_PROTEINS))
annotation_query_complexes = annotation_query_complexes.replace('LIST', str(BACKGROUND_PROTEINS))

### And "send" the query to the database:

In [ ]:
driver = connector.getGraphDatabaseConnectionConfiguration()
annotation_bp = connector.getCursorData(driver, annotation_query_bp)
annotation_mf = connector.getCursorData(driver, annotation_query_mf)
annotation_pathway = connector.getCursorData(driver, annotation_query_pathway)
annotation_complexes = connector.getCursorData(driver, annotation_query_complexes)

In [ ]:
annotation_drug = connector.getCursorData(driver, annotation_query_drug)

In [ ]:
annotation_disease = connector.getCursorData(driver, annotation_query_disease)

### Drug associations

In [ ]:
DRUG = 'Metformin'
ANNOTATION = annotation_drug
REGULATED_PROTEINS = files_dict[DRUG]['Gene name'].unique().tolist()
regulated=ANNOTATION[ANNOTATION['identifier'].isin(REGULATED_PROTEINS)].sort_values(by='annotation')
regulated[regulated['annotation']==DRUG]

#### Enrichment 

In [ ]:
foreground_list=REGULATED_PROTEINS
background_list=BACKGROUND_PROTEINS
annotation = ANNOTATION

In [ ]:
print(len(background_list), len(foreground_list))

In [ ]:
enrichment_result = run_enrichment(annotation, 
                                   foreground_list=foreground_list, 
                                   background_list=background_list, 
                                   foreground_pop=len(foreground_list), 
                                   background_pop=len(background_list))

ANNOTATION[ANNOTATION['annotation']==DRUG].sort_values('identifier').head()
#enrichment_result.to_csv('projects/xxx/enrichment_drug.csv')

In [ ]:
enrichment_result[enrichment_result['terms']==DRUG]

### Disease associations

In [ ]:
DRUG = 'Metformin'
ANNOTATION = annotation_disease
REGULATED_PROTEINS = files_dict[DRUG]['Gene name'].unique().tolist()
regulated=ANNOTATION[ANNOTATION['identifier'].isin(REGULATED_PROTEINS)].sort_values(by='annotation')

In [ ]:
regulated.sort_values('annotation').head()

In [ ]:
regulated.shape

#### Enrichment 

In [ ]:
foreground_list=REGULATED_PROTEINS
background_list=BACKGROUND_PROTEINS
annotation = ANNOTATION
print(len(background_list), len(foreground_list))

In [ ]:
annotation[annotation['annotation']=='diabetes mellitus'].shape

In [ ]:
regulated_disease = annotation[annotation['identifier'].isin(REGULATED_PROTEINS)]
regulated_disease[regulated_disease['annotation']=='diabetes mellitus'].head()

In [ ]:
enrichment_result = run_enrichment(annotation, 
                                   foreground_list=foreground_list, 
                                   background_list=background_list, 
                                   foreground_pop=len(foreground_list), 
                                   background_pop=len(background_list))
enrichment_result.head()
#enrichment_result.to_csv('projects/xxx/enrichment_disease.csv')

### Functional annotations

In [ ]:
DRUG = 'Metformin'
ANNOTATION = annotation_pathway
REGULATED_PROTEINS = files_dict[DRUG]['Gene name'].unique().tolist()
regulated=ANNOTATION[ANNOTATION['identifier'].isin(REGULATED_PROTEINS)].sort_values(by='annotation')

In [ ]:
regulated.sort_values('annotation').head()

In [ ]:
regulated.shape

#### Enrichment 

In [ ]:
foreground_list=REGULATED_PROTEINS
background_list=BACKGROUND_PROTEINS
annotation = ANNOTATION
print(len(background_list), len(foreground_list))

In [ ]:
enrichment_result = run_enrichment(annotation, 
                                   foreground_list=foreground_list, 
                                   background_list=background_list, 
                                   foreground_pop=len(foreground_list), 
                                   background_pop=len(background_list))
enrichment_result
enrichment_result.to_csv('projects/xxx/enrichment_pathway.csv')

In [ ]:
import scipy.stats as stats

In [ ]:
stats.fisher_exact([[2, 44], [1, 213]])